In [29]:
import os 
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, classification_report
from tqdm.notebook import tqdm

import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
label_columns = ['normal', 'pneumonia', 'chf']
image_size = 224 
image_dir = '/kaggle/input/processed-data/images'
concepts_dir = '/kaggle/input/final-concepts/FINAL_CONCEPT(latest) (1).csv'
aug_file = "/kaggle/working/augmented_concepts.csv"

In [3]:
def combine_labels(row):
    for idx, col in enumerate(label_columns):
        if row[col] == 1:
            return idx
    return -1

In [4]:
def is_text_column(series, col):
    if col in ['id', 'label']:
        return False
    return series.apply(lambda x: isinstance(x, str)).any()

In [5]:
concepts_df = pd.read_csv(concepts_dir)

if 'sid' in concepts_df.columns:
    concepts_df = concepts_df.drop(columns=['sid'])

concepts_df['label'] = concepts_df.apply(combine_labels, axis=1)
concepts_df = concepts_df.drop(columns=label_columns)

In [6]:
text_columns = [col for col in concepts_df.columns if is_text_column(concepts_df[col], col)]
concepts_df = concepts_df.drop(columns=text_columns)

In [7]:
concepts_df['id'] = concepts_df['id'].astype(str) + "_0"

In [8]:
augmented_files = [f for f in os.listdir(image_dir) if '_0.' not in f]

augmented_rows = []

concept_dict = concepts_df.set_index('id').to_dict(orient='index')

for filename in tqdm(augmented_files):
    base_id = filename.split('_')[0]  
    original_id = base_id + '_0'

    if original_id in concept_dict:
        new_row = concept_dict[original_id].copy()
        new_row['id'] = filename.replace('.png', '')  
        augmented_rows.append(new_row)

augmented_df = pd.DataFrame(augmented_rows)

full_concepts_df = pd.concat([concepts_df, augmented_df], ignore_index=True)

full_concepts_df.to_csv(aug_file, index=False)

print(f"Final dataset has {len(full_concepts_df)} rows.")

del augmented_df, full_concepts_df

  0%|          | 0/15480 [00:00<?, ?it/s]

Final dataset has 16477 rows.


In [9]:
class XrayConceptDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.copy()
        self.image_dir = image_dir
        self.transform = transform

        self.df = self.df[self.df['id'].apply(lambda x: os.path.exists(os.path.join(image_dir, f"{x}.png")))]
        self.df = self.df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = f"{row['id']}.png"
        image_path = os.path.join(self.image_dir, image_id)

        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        concepts = torch.tensor(row.drop(['id', 'label']).astype(float).values, dtype=torch.float32)

        return image, concepts

In [10]:
transform = T.Compose([
    T.Resize((image_size, image_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

In [11]:
df = pd.read_csv(aug_file)
dataset = XrayConceptDataset(dataframe=df, image_dir=image_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

In [12]:
class ViTConceptPredictor(nn.Module):
    def __init__(self, vit_name='vit_base_patch16_224', num_concepts=314):
        super().__init__()
        self.backbone = timm.create_model(vit_name, pretrained=True, num_classes=0)
        self.head = nn.Linear(self.backbone.num_features, num_concepts)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

In [13]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = XrayConceptDataset(train_df, image_dir, transform=transform)
val_dataset = XrayConceptDataset(val_df, image_dir, transform=transform)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTConceptPredictor().to(device)
criterion = nn.BCEWithLogitsLoss()  
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [15]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets.numpy())

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    binary_preds = (all_preds > 0.5).astype(int)

    f1s = f1_score(all_targets, binary_preds, average=None)
    avg_f1 = f1_score(all_targets, binary_preds, average="macro")

    roc_aucs = []
    for i in range(all_targets.shape[1]): 
        if len(np.unique(all_targets[:, i])) == 1:
            roc_aucs.append(np.nan)  
        else:
            auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
            roc_aucs.append(auc)

    avg_roc_auc = np.nanmean(roc_aucs)  

    accuracy = accuracy_score(all_targets.flatten(), binary_preds.flatten())

    print(f"Macro F1 Score: {avg_f1:.4f}")
    print(f"Avg ROC AUC Score: {avg_roc_auc:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    
    return f1s, avg_f1, roc_aucs, avg_roc_auc, accuracy

In [ ]:
epochs = 5
for epoch in range(epochs):
    model.train()
    total_train_loss = 0

    for images, targets in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}/{epochs}", ncols=100, leave=False):
        images, targets = images.to(device), targets.to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"\nEpoch [{epoch+1}/{epochs}], Training Loss: {avg_train_loss:.4f}")

    print("\nEvaluating on Training Set...")
    train_f1s, avg_train_f1, train_roc_aucs, avg_train_roc_auc, train_accuracy = evaluate(model, train_dataloader, device)

    print("\nEvaluating on Validation Set...")
    for images, targets in tqdm(val_dataloader, desc=f"Validating Epoch {epoch+1}/{epochs}", ncols=100, leave=False):
        images, targets = images.to(device), targets.to(device)
    val_f1s, avg_val_f1, val_roc_aucs, avg_val_roc_auc, val_accuracy = evaluate(model, val_dataloader, device)

    print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f}, "
          f"Train F1: {avg_train_f1:.4f}, Train ROC AUC: {avg_train_roc_auc:.4f}, Train Accuracy: {train_accuracy:.4f}, "
          f"Val F1: {avg_val_f1:.4f}, Val ROC AUC: {avg_val_roc_auc:.4f}, Val Accuracy: {val_accuracy:.4f}")


In [17]:
model_save_path = "images_concepts_model_weights.pth"
torch.save(model.state_dict(), f"vit_model_weights_epoch_{epoch+1}.pth")

In [18]:
vit_model = ViTConceptPredictor()

In [19]:
vit_model.load_state_dict(torch.load(
    "/kaggle/input/images-concepts-wts/pytorch/default/1/vit_model_weights_epoch_5.pth", 
    map_location=torch.device('cpu')
))

/tmp/ipykernel_107/3435040439.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vit_model.load_state_dict(torch.load(


<All keys matched successfully>

In [20]:
for param in vit_model.parameters():
    param.requires_grad = False  
vit_model.to(device)

ViTConceptPredictor(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approxima

In [21]:
class ConceptToLabelClassifier(nn.Module):
    def __init__(self, input_dim=314, hidden_dim=512, num_classes=3):
        super(ConceptToLabelClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)  
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [22]:
class ConceptOnlyMultiClassDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        concept_vector = torch.tensor(row.iloc[1:-1].values.astype(np.float32))  
        label = torch.tensor(int(row['label']), dtype=torch.long)  
        return concept_vector, label

In [23]:
train_dataset = ConceptOnlyMultiClassDataset(train_df)
val_dataset = ConceptOnlyMultiClassDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [24]:
concept_to_label_model = ConceptToLabelClassifier().to(device)
optimizer = torch.optim.Adam(concept_to_label_model.parameters(), lr=1e-4)
class_weights = torch.tensor([2.0, 2.0, 1.0]).to(device) 
criterion = nn.CrossEntropyLoss(weight=class_weights)
epochs = 20

In [25]:
for epoch in range(epochs):
    vit_model.eval()  
    concept_to_label_model.train()  
    total_loss = 0
    train_preds, train_labels = [], []

    for concepts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", ncols=100):
        concepts, labels = concepts.to(device), labels.to(device)

        outputs = concept_to_label_model(concepts)  
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(labels.cpu().numpy())

    train_accuracy = accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds, average='weighted')
    train_precision = precision_score(train_labels, train_preds, average='weighted')
    train_recall = recall_score(train_labels, train_preds, average='weighted')
    train_class_report = classification_report(train_labels, train_preds, output_dict=True)

    print(f"Epoch [{epoch+1}/{epochs}] - Training Loss: {total_loss / len(train_loader):.4f}")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Training Precision: {train_precision:.4f}")
    print(f"Training Recall: {train_recall:.4f}")
    print(f"Training F1 Score: {train_f1:.4f}")
    
    print(f"Classwise Accuracy (Training):")
    for i in range(len(train_class_report) - 3): 
        print(f"Class {i}: {train_class_report[str(i)]['f1-score']:.4f}")

    concept_to_label_model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for concepts, labels in val_loader:
            outputs = concept_to_label_model(concepts.to(device))
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_accuracy = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    val_precision = precision_score(all_labels, all_preds, average='weighted')
    val_recall = recall_score(all_labels, all_preds, average='weighted')
    val_class_report = classification_report(all_labels, all_preds, output_dict=True)

    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Validation Precision: {val_precision:.4f}")
    print(f"Validation Recall: {val_recall:.4f}")
    print(f"Validation F1 Score: {val_f1:.4f}")
    
    print(f"Classwise Accuracy (Validation):")
    for i in range(len(val_class_report) - 3):  
        print(f"Class {i}: {val_class_report[str(i)]['f1-score']:.4f}")

print(classification_report(train_labels, train_preds, digits=4))

Epoch 1:   0%|                                                              | 0/412 [00:00<?, ?it/s]

Epoch [1/20] - Training Loss: 0.6581
Training Accuracy: 0.7182
Training Precision: 0.7206
Training Recall: 0.7182
Training F1 Score: 0.7145
Classwise Accuracy (Training):
Class 0: 0.7844
Class 1: 0.6217
Class 2: 0.7344
Validation Accuracy: 0.8225
Validation Precision: 0.8359
Validation Recall: 0.8225
Validation F1 Score: 0.8205
Classwise Accuracy (Validation):
Class 0: 0.8419
Class 1: 0.7571
Class 2: 0.8600


Epoch 2:   0%|                                                              | 0/412 [00:00<?, ?it/s]

Epoch [2/20] - Training Loss: 0.4689
Training Accuracy: 0.8157
Training Precision: 0.8255
Training Recall: 0.8157
Training F1 Score: 0.8150
Classwise Accuracy (Training):
Class 0: 0.8316
Class 1: 0.7560
Class 2: 0.8544
Validation Accuracy: 0.8832
Validation Precision: 0.9011
Validation Recall: 0.8832
Validation F1 Score: 0.8836
Classwise Accuracy (Validation):
Class 0: 0.8703
Class 1: 0.8452
Class 2: 0.9353


Epoch 3:   0%|                                                              | 0/412 [00:00<?, ?it/s]

...

Epoch [18/20] - Training Loss: 0.2129
Training Accuracy: 0.9162
Training Precision: 0.9250
Training Recall: 0.9162
Training F1 Score: 0.9163
Classwise Accuracy (Training):
Class 0: 0.8909
Class 1: 0.8775
Class 2: 0.9774
Validation Accuracy: 0.9257
Validation Precision: 0.9375
Validation Recall: 0.9257
Validation F1 Score: 0.9257
Classwise Accuracy (Validation):
Class 0: 0.9040
Class 1: 0.8907
Class 2: 0.9827


Epoch 19:   0%|                                                             | 0/412 [00:00<?, ?it/s]

Epoch [19/20] - Training Loss: 0.2123
Training Accuracy: 0.9168
Training Precision: 0.9267
Training Recall: 0.9168
Training F1 Score: 0.9170
Classwise Accuracy (Training):
Class 0: 0.8896
Class 1: 0.8784
Class 2: 0.9796
Validation Accuracy: 0.9214
Validation Precision: 0.9310
Validation Recall: 0.9214
Validation F1 Score: 0.9214
Classwise Accuracy (Validation):
Class 0: 0.8988
Class 1: 0.8844
Class 2: 0.9814


Epoch 20:   0%|                                                             | 0/412 [00:00<?, ?it/s]

Epoch [20/20] - Training Loss: 0.2084
Training Accuracy: 0.9179
Training Precision: 0.9276
Training Recall: 0.9179
Training F1 Score: 0.9181
Classwise Accuracy (Training):
Class 0: 0.8913
Class 1: 0.8806
Class 2: 0.9792
Validation Accuracy: 0.9251
Validation Precision: 0.9365
Validation Recall: 0.9251
Validation F1 Score: 0.9249
Classwise Accuracy (Validation):
Class 0: 0.9043
Class 1: 0.8896
Class 2: 0.9814
              precision    recall  f1-score   support

           0     0.8194    0.9771    0.8913      4374
           1     0.9661    0.8090    0.8806      4267
           2     0.9957    0.9632    0.9792      4540

    accuracy                         0.9179     13181
   macro avg     0.9271    0.9165    0.9170     13181
weighted avg     0.9276    0.9179    0.9181     13181



In [32]:
image_path = "/kaggle/input/processed-data/images/002da0d9-ce49c30d-4dfcc1f8-746d2401-d8044d48_10.png"  #image path
image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

input_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    concept_outputs = model(input_tensor)  
    probs = torch.sigmoid(logits).cpu().numpy().flatten() 

# Convert to NumPy array
concepts = concept_outputs.squeeze().cpu().numpy()
print("Concept predictions:")
print(concepts[:])

binary_preds = (probs >= 0.5).astype(int)
print("[", " ".join(map(str, binary_preds)), "]")


Concept predictions:
[-11.682538   -8.566617  -10.240375   -6.4375453  -5.5624237  -9.584036
  -7.7453246  -9.261873   -5.4589167 -10.15657    -8.681663   -6.304719
  -9.503451  -10.200921   -8.813296   -9.403074   -9.929661   -8.273896
 -10.282324   -8.738947   -8.466653   -8.63791    -9.12493    -8.768427
  -9.088313  -10.538927   -9.109181   -8.5846615  -7.692397  -10.922863
  -8.895763  -11.182313  -10.736176   -9.684544   -9.549604   -6.714327
  -8.015347  -11.444966   -8.906038   -7.911041  -10.422622   -8.115451
  -8.922825   -9.684103   -5.38702    -9.091316    6.6720986 -10.29561
  -9.703041  -11.01747    -9.682447   -8.047309   -9.001955   -5.571136
 -10.116871   -9.902333   -8.560713   -9.425264   -7.823977  -11.742486
  -7.3857636  -7.087965  -11.241545   -9.544121   -9.651269  -11.061626
  -9.545521    2.7686646  -9.183349   -8.723585  -11.47435    -9.065539
  -8.017532   -8.543116   -7.9265566  -7.476787   -8.917025   -8.500277
  -8.978445   -8.379047  -10.178992   -8.597